In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
t2012 = pd.read_csv("Tabela5-sem_emprego_2012.csv", sep=";", decimal=",")
t2026 = pd.read_csv("Tabela5-sem_emprego_2026.csv", sep=";", decimal=",")

t2012 = t2012[["Sigla", "Código", "Estado", "Desocupados - mulheres (2012 T1)"]]
t2026 = t2026[["Sigla", "Código", "Estado", "Desocupados - mulheres (2026 T1)"]]

t2012 = t2012.rename(columns={"Desocupados - mulheres (2012 T1)": "mulheres_2012"})
t2026 = t2026.rename(columns={"Desocupados - mulheres (2026 T1)": "mulheres_2026"})

In [ ]:
indicador_1 = pd.read_excel(
    "Tabela 1.1.1 (1).xls",
    sheet_name="2022",
    skiprows=8,
    header=None,
    usecols=[0, 6, 7],
)
indicador_1.columns = ["Estado", "mulher_branca", "mulher_preta_parda"]
indicador_1 = indicador_1[indicador_1["Estado"].isin(t2012["Estado"])].copy()
indicador_1["mulheres"] = (
    pd.to_numeric(indicador_1["mulher_branca"])
    + pd.to_numeric(indicador_1["mulher_preta_parda"])
) / 2
indicador_1 = indicador_1[["Estado", "mulheres"]]

In [ ]:
comp = t2012.merge(t2026, on=["Sigla", "Código", "Estado"], how="inner")
comp = comp.merge(indicador_1, on="Estado", how="inner")

In [ ]:
ordem = comp.sort_values("mulheres_2026")["Estado"]
longo = comp.melt(
    id_vars=["Estado", "mulheres"],
    value_vars=["mulheres_2012", "mulheres_2026"],
    var_name="Ano",
    value_name="Participacao_mulheres",
)
longo["Ano"] = longo["Ano"].map({
    "mulheres_2012": "2012 T1",
    "mulheres_2026": "2026 T1",
})

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
sns.barplot(
    data=longo,
    y="Estado",
    x="Participacao_mulheres",
    hue="Ano",
    order=ordem,
    ax=ax,
)
ax.axvline(50, color="gray", linestyle="--")
ax.set_xlabel("Participação das mulheres entre as pessoas desocupadas (%)")
ax.set_ylabel("")
ax.set_title("Desocupação: participação feminina em 2012 T1 e 2026 T1")
ax.legend(title="Ano")
fig.tight_layout()
plt.show()